In [ ]:
import pandas as pd
df = pd.read_csv("../data/PIK3CA_ecfp2048.csv")
df

In [ ]:
df2 = df.drop("valid_smiles", axis = 1)

In [ ]:
df2.head()

In [ ]:
x = df2.drop("pIC50", axis = 1)
y = df2["pIC50"]

In [ ]:
x

In [ ]:
y

In [ ]:
import warnings, time
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.base import clone
try:
    from IPython.display import display
except ImportError:
    def display(*a, **k):
        print(*a)

warnings.filterwarnings("ignore")

from sklearn.linear_model import (SGDRegressor, LassoLars, ElasticNet, Lasso,
    QuantileRegressor, PassiveAggressiveRegressor, LinearRegression, Ridge,
    HuberRegressor, RidgeCV, Lars, LarsCV, OrthogonalMatchingPursuit,
    OrthogonalMatchingPursuitCV, LassoLarsIC, LassoLarsCV, GammaRegressor,
    TweedieRegressor, PoissonRegressor, BayesianRidge, LassoCV, ElasticNetCV,
    RANSACRegressor)
from sklearn.kernel_ridge import KernelRidge
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import LinearSVR, NuSVR, SVR
from sklearn.dummy import DummyRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
    GradientBoostingRegressor, HistGradientBoostingRegressor,
    AdaBoostRegressor, BaggingRegressor)

MODEL_FACTORY = {
    'SGDRegressor': SGDRegressor, 'KernelRidge': KernelRidge,
    'GaussianProcessRegressor': GaussianProcessRegressor,
    'RANSACRegressor': RANSACRegressor, 'MLPRegressor': MLPRegressor,
    'LassoLars': LassoLars, 'DummyRegressor': DummyRegressor,
    'ElasticNet': ElasticNet, 'Lasso': Lasso,
    'QuantileRegressor': QuantileRegressor,
    'AdaBoostRegressor': AdaBoostRegressor, 'LinearSVR': LinearSVR,
    'PassiveAggressiveRegressor': PassiveAggressiveRegressor,
    'LinearRegression': LinearRegression,
    'TransformedTargetRegressor': TransformedTargetRegressor,
    'Ridge': Ridge, 'HuberRegressor': HuberRegressor, 'RidgeCV': RidgeCV,
    'Lars': Lars, 'LarsCV': LarsCV,
    'ExtraTreeRegressor': ExtraTreeRegressor,
    'DecisionTreeRegressor': DecisionTreeRegressor,
    'KNeighborsRegressor': KNeighborsRegressor,
    'GradientBoostingRegressor': GradientBoostingRegressor,
    'ExtraTreesRegressor': ExtraTreesRegressor,
    'OrthogonalMatchingPursuit': OrthogonalMatchingPursuit,
    'OrthogonalMatchingPursuitCV': OrthogonalMatchingPursuitCV,
    'LassoLarsIC': LassoLarsIC, 'LassoLarsCV': LassoLarsCV,
    'GammaRegressor': GammaRegressor, 'TweedieRegressor': TweedieRegressor,
    'NuSVR': NuSVR, 'SVR': SVR, 'PoissonRegressor': PoissonRegressor,
    'BayesianRidge': BayesianRidge, 'LassoCV': LassoCV,
    'ElasticNetCV': ElasticNetCV,
    'HistGradientBoostingRegressor': HistGradientBoostingRegressor,
    'BaggingRegressor': BaggingRegressor,
    'RandomForestRegressor': RandomForestRegressor,
}
try:  
    from lightgbm import LGBMRegressor
    MODEL_FACTORY['LGBMRegressor'] = LGBMRegressor
except Exception:
    print("lightgbm not installed; LGBMRegressor skipped this run")


EXCLUDE = {'GaussianProcessRegressor', 'RANSACRegressor', 'KernelRidge', 'SGDRegressor'}
active = [n for n in MODEL_FACTORY if n not in EXCLUDE]
print(f"benchmarking {len(active)} models with plain 5-fold CV (excluded: {sorted(EXCLUDE)})")


N_FOLDS, SEED = 5, 42
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def _prepare(est):
    """Fix n_jobs and random_state for speed and reproducibility."""
    try:
        p = est.get_params()
        if 'n_jobs' in p:
            est.set_params(n_jobs=-1)
        if 'random_state' in p:
            est.set_params(random_state=SEED)
    except Exception:
        pass
    return est

rows, t0_all = [], time.time()
for k, name in enumerate(active, 1):
    r2s, rmses, t0 = [], [], time.time()
    try:
        for tr_idx, te_idx in kf.split(x):
            est = _prepare(clone(MODEL_FACTORY[name]()))
            est.fit(x.iloc[tr_idx], y.iloc[tr_idx])
            y_true_fold = y.iloc[te_idx]
            y_pred_fold = est.predict(x.iloc[te_idx])
            r2s.append(r2_score(y_true_fold, y_pred_fold))
            rmses.append(float(np.sqrt(mean_squared_error(y_true_fold, y_pred_fold))))
        rows.append({
            'Model': name,
            'R-Squared': float(np.mean(r2s)),
            'R2_SD': float(np.std(r2s, ddof=1)),
            'RMSE': float(np.mean(rmses)),
            'RMSE_SD': float(np.std(rmses, ddof=1)),
            'Time Taken': round(time.time() - t0, 1),
        })
        r = rows[-1]
        print(f"[{k:>2}/{len(active)}] {name:<32s} "
              f"R2={r['R-Squared']:.4f}+/-{r['R2_SD']:.4f}  "
              f"RMSE={r['RMSE']:.4f}+/-{r['RMSE_SD']:.4f}  ({r['Time Taken']:.0f}s)",
              flush=True)
    except Exception as e:
        print(f"[{k:>2}/{len(active)}] {name:<32s} failed: {type(e).__name__}: {e}", flush=True)
  
    pd.DataFrame(rows).to_csv("screening_5fold_partial.csv", index=False)

models_df = (pd.DataFrame(rows).set_index('Model')
           .sort_values('R-Squared', ascending=False))
）
predictions_df = models_df.copy()

models_df.to_csv("screening_5fold_results.csv")  
print(f"\ndone in {time.time() - t0_all:.0f} s; results in screening_5fold_results.csv")
display(models_df)

In [ ]:
display(models_df)

In [ ]:
predictions_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns



# Horizontal bar chart of benchmark results
plt.figure(figsize=(8, 6))

# Plot styling
sns.set_theme(style="whitegrid")


plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12


ax = sns.barplot(
    y=predictions_df.index,
    x="R-Squared",
    data=predictions_df,
    palette="Blues_d",          
    edgecolor='black',          
    linewidth=0.5
)

# Axis limits
ax.set(xlim=(0, 1))

# Axis labels
ax.set_xlabel('R² Score', fontsize=14, fontweight='bold')
ax.set_ylabel('Model', fontsize=14, fontweight='bold')  


plt.tight_layout()

# Save as high-resolution PNG and PDF
plt.savefig('PIK3CA_r_squared_comparison-1.pdf', dpi=300, bbox_inches='tight')
plt.savefig('PIK3CA_r_squared_comparison-1.png', dpi=300, bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns




plt.figure(figsize=(8, 6))

# Plot styling
sns.set_theme(style="whitegrid")

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12


ax = sns.barplot(
    y=predictions_df.index,
    x="RMSE",
    data=predictions_df,
    palette="Blues_r",          
    edgecolor='black',         
    linewidth=0.5
)

# Axis limits
ax.set(xlim=(0, 1.5))

# Axis labels
ax.set_xlabel('RMSE', fontsize=14, fontweight='bold')
ax.set_ylabel('Model', fontsize=14, fontweight='bold')  


# Layout
plt.tight_layout()

# Save as PNG and PDF
plt.savefig('PIK3CA_rmse_comparison-1.pdf', dpi=300, bbox_inches='tight')
plt.savefig('PIK3CA_rmse_comparison-1.png', dpi=300, bbox_inches='tight')